## Introduction
This notebook trains several machine learning models to predict close price from a property's physical and geographic features and evaluates their performance on the validation set.

In [1]:
import numpy as np
import pandas as pd

In [2]:
train = pd.read_csv('clean-data/sold_train.csv')
val = pd.read_csv('clean-data/sold_validation.csv')
test = pd.read_csv('clean-data/sold_test.csv')

## Part 1: Setup
#### 1.1 Load machine learning libraries and preprocessing pipeline

In [3]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from category_encoders import TargetEncoder
from utilities import impute_groupwise, get_preprocessor
from sklearn import set_config

In [4]:
# force every transformer to output clean pandas dataframes instead of raw numpy arrays
set_config(transform_output='pandas')

#### 1.2 Initiate performance tracker
Create a class to compute, log, and manage evaluation metrics across multiple machine learning models.

In [5]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [6]:
class ModelPerformanceTracker:
    # set tracker schema
    def __init__(self):
        self.summary = pd.DataFrame(columns=[
            'Model', 'Feature Size', 'R2', 'RMSE ($)', 'MAE ($)', 'MAPE (%)', 'MdAPE (%)'
        ])

    # calculate evaluation metrics and append them to the summary table
    def log_results(self, model_name, feature_size, y_true, y_pred):
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)

        eps = 1e-8 # have an epsilon in place to prevent division by 0
        percentage_errors = np.abs((y_true - y_pred) / (y_true + eps)) * 100
        mape = np.mean(percentage_errors)
        mdape = np.median(percentage_errors)
        
        model_results = pd.DataFrame([{
            'Model': model_name,
            'Feature Size': feature_size,
            'R2': round(r2, 4),
            'RMSE ($)': round(rmse, 2),
            'MAE ($)': round(mae, 2), 
            'MAPE (%)': round(mape, 2),
            'MdAPE (%)': round(mdape, 2)
        }])
        self.summary = pd.concat([self.summary, model_results], ignore_index=True)

    # return model performance summary sorted by MdAPE  
    def get_summary(self):
        return self.summary.sort_values(by='MdAPE (%)', ascending=True).reset_index(drop=True)

In [7]:
# initiate the tracker
tracker = ModelPerformanceTracker()

## Part 2: Modeling with Original Features
Train models using the original features in the training set.

In [8]:
location_cols = ['MLSAreaMajor', 'CountyOrParish', 'City', 'PostalCode']
bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']
numeric_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 
                'LotSizeSquareFeet', 'ParkingTotal', 'Stories', 'YearBuilt', 'AssociationFee']
X = location_cols + bool_cols + numeric_cols

In [9]:
X_train = train[X]
y_train = train['ClosePrice']

X_val = val[X]
y_val = val['ClosePrice']

X_test = test[X]
y_test = test['ClosePrice']

#### 2.1 Regression Models
Build linear regression models with a log-transformed target, beginning with ordinary least squares (OLS) and then extending to ridge regression.

In [10]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import SplineTransformer

In [11]:
linear_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=True
)

In [12]:
ols_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', LinearRegression())
])

ols_model = TransformedTargetRegressor(
    regressor=ols_pipeline, func=np.log1p, inverse_func=np.expm1
)

ols_model.fit(X_train, y_train)
ols_pred_val = ols_model.predict(X_val)
tracker.log_results(
    model_name='Linear Regression', 
    feature_size=len(X),
    y_true=y_val, 
    y_pred=ols_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [13]:
ridge_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('estimator', Ridge(alpha=1.0)) # L2 penalty
])

ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipeline, func=np.log1p, inverse_func=np.expm1
)

ridge_model.fit(X_train, y_train)
ridge_pred_val = ridge_model.predict(X_val)
tracker.log_results(
    model_name='Ridge Regression',
    feature_size=len(X),
    y_true=y_val,
    y_pred=ridge_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

Given that regularization does not significantly improve the performance of linear regression models, try natural splines to introduce non-linearity.

In [14]:
n_knots = 5
degree = 3 # natural splines use cubic polynomials

In [15]:
spline_transformer = ColumnTransformer(
    transformers=[
        ('splines', 
         SplineTransformer(
             n_knots=n_knots, 
             degree=degree, 
             extrapolation='linear', # enforce linear boundaries
             include_bias=False),    # exclude intercept to reduce multi-collinearity
         make_column_selector(dtype_include=['float64', 'float32', 'int64', 'int32']))
    ],
    remainder='passthrough' 
)

spline_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('natural_splines', spline_transformer),
    ('estimator', LinearRegression())  
])

spline_model = TransformedTargetRegressor(
    regressor=spline_pipeline, func=np.log1p, inverse_func=np.expm1
)

In [16]:
spline_model.fit(X_train, y_train)
spline_pred_val = spline_model.predict(X_val)

tracker.log_results(
    model_name=f'Natural Spline ({n_knots} Knots)',
    feature_size=len(X),
    y_true=y_val,
    y_pred=spline_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [17]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,Natural Spline (5 Knots),17,0.8374,376332.44,204533.75,15.54,11.52
1,Linear Regression,17,0.7987,418770.52,219636.26,16.56,12.42
2,Ridge Regression,17,0.7987,418773.95,219637.29,16.56,12.42


#### 2.2 Tree-Based Models
Build tree-based models with a log-transformed target, beginning with decision tree and then extending to random forest.

In [18]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [19]:
tree_preprocessor = get_preprocessor(
    location_cols, 
    bool_cols, 
    numeric_cols,
    scale_skewed=False # no need to normalize numeric features for tree architectures
)

Decision trees:

In [20]:
depth = 12

In [21]:
dt_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', DecisionTreeRegressor(max_depth=depth, random_state=42))
])

dt_model = TransformedTargetRegressor(
    regressor=dt_pipeline, func=np.log1p, inverse_func=np.expm1
)

dt_model.fit(X_train, y_train)
dt_pred_val = dt_model.predict(X_val)

# Log results to your existing performance tracker
tracker.log_results(
    model_name=f'Decision Tree (Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=dt_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

Random forests:

In [22]:
n_trees = 150
depth = 12

In [23]:
rf_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=42, n_jobs=-1))
])

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
)

rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
tracker.log_results(
    model_name=f'Random Forest ({n_trees} Trees, Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=rf_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [24]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (150 Trees, Depth=12)",17,0.8693,337452.32,178084.37,13.06,9.39
1,Decision Tree (Depth=12),17,0.8346,379572.76,197266.26,14.59,10.36
2,Natural Spline (5 Knots),17,0.8374,376332.44,204533.75,15.54,11.52
3,Linear Regression,17,0.7987,418770.52,219636.26,16.56,12.42
4,Ridge Regression,17,0.7987,418773.95,219637.29,16.56,12.42


## Part 3: Modeling with Engineered Features
Add property age and school district to the feature set.

In [25]:
location_cols = ['MLSAreaMajor', 'CountyOrParish', 'City', 'PostalCode', 'DistrictNa']
bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']
numeric_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 
                'LotSizeSquareFeet', 'ParkingTotal', 'Stories', 'AssociationFee', 'YearBuilt',
                'property_age', 'floor_area_ratio']
X = location_cols + bool_cols + numeric_cols

In [26]:
X_train = train[X]
y_train = train['ClosePrice']

X_val = val[X]
y_val = val['ClosePrice']

X_test = test[X]
y_test = test['ClosePrice']

#### 3.1 Regression Models
Train a natural spline model with log-transformed target.

In [27]:
n_knots = 5
degree = 3

In [28]:
spline_transformer = ColumnTransformer(
    transformers=[
        ('splines', 
         SplineTransformer(
             n_knots=n_knots, 
             degree=degree, 
             extrapolation='linear', # enforce linear boundaries
             include_bias=False),    # exclude intercept to reduce multi-collinearity
         make_column_selector(dtype_include=['float64', 'float32', 'int64', 'int32']))
    ],
    remainder='passthrough' 
)

spline_pipeline = Pipeline([
    ('features', linear_preprocessor),
    ('natural_splines', spline_transformer),
    ('estimator', LinearRegression())  
])

spline_model = TransformedTargetRegressor(
    regressor=spline_pipeline, func=np.log1p, inverse_func=np.expm1
)

In [29]:
spline_model.fit(X_train, y_train)
spline_pred_val = spline_model.predict(X_val)

tracker.log_results(
    model_name=f'Natural Spline ({n_knots} Knots)',
    feature_size=len(X),
    y_true=y_val,
    y_pred=spline_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [30]:
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (150 Trees, Depth=12)",17,0.8693,337452.32,178084.37,13.06,9.39
1,Decision Tree (Depth=12),17,0.8346,379572.76,197266.26,14.59,10.36
2,Natural Spline (5 Knots),17,0.8374,376332.44,204533.75,15.54,11.52
3,Natural Spline (5 Knots),20,0.8374,376332.44,204533.75,15.54,11.52
4,Linear Regression,17,0.7987,418770.52,219636.26,16.56,12.42
5,Ridge Regression,17,0.7987,418773.95,219637.29,16.56,12.42


#### 3.2 Tree-Based Models
Train a random forest with log-transformed target.

In [31]:
n_trees = 150
depth = 12

In [32]:
rf_pipeline = Pipeline([
    ('features', tree_preprocessor),
    ('estimator', RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=42, n_jobs=-1))
])

rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
)

rf_model.fit(X_train, y_train)
rf_pred_val = rf_model.predict(X_val)
tracker.log_results(
    model_name=f'Random Forest ({n_trees} Trees, Depth={depth})',
    feature_size=len(X),
    y_true=y_val,
    y_pred=rf_pred_val
)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/wenxincui/Library/Python/3.9/lib/

In [33]:
# TODO: new features did not bring noticeable performance improvement
tracker.get_summary()

,Model,Feature Size,R2,RMSE ($),MAE ($),MAPE (%),MdAPE (%)
0,"Random Forest (150 Trees, Depth=12)",17,0.8693,337452.32,178084.37,13.06,9.39
1,"Random Forest (150 Trees, Depth=12)",20,0.8693,337452.32,178084.37,13.06,9.39
2,Decision Tree (Depth=12),17,0.8346,379572.76,197266.26,14.59,10.36
3,Natural Spline (5 Knots),17,0.8374,376332.44,204533.75,15.54,11.52
4,Natural Spline (5 Knots),20,0.8374,376332.44,204533.75,15.54,11.52
5,Linear Regression,17,0.7987,418770.52,219636.26,16.56,12.42
6,Ridge Regression,17,0.7987,418773.95,219637.29,16.56,12.42
